In [ ]:
import pandas as pd

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/CSV_main")

matches = pd.read_csv(DATA_DIR / "matches.csv")
deliveries = pd.read_csv(DATA_DIR / "deliveries.csv")

In [ ]:
matches_df = pd.DataFrame(matches)
deliveries_df = pd.DataFrame(deliveries)

In [ ]:
print('Matches columns:')
print(matches_df.columns.to_list())

In [ ]:
print('Deliveries columns:')
print(deliveries_df.columns.to_list())

In [ ]:
print(matches_df['date'].dtype)

In [ ]:
matches_df['date'] = pd.to_datetime(matches_df['date'])

In [ ]:
print(matches_df['date'].dtype)

In [ ]:
print(matches_df.isnull().sum())
print('--------')
print(deliveries_df.isnull().sum())

In [ ]:
matches_df.dropna(subset=['winner'] , inplace=True)


In [ ]:
matches_df.drop(columns=['city', 'method', 'umpire1', 'umpire2'] , inplace = True)

In [ ]:
team_wins_df = matches_df['winner'].value_counts().rename_axis('Team').reset_index(name="Wins").sort_values(by='Wins', ascending = False)

In [ ]:
team_wins_df

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.barh(team_wins_df.Team ,team_wins_df.Wins)
plt.title("Who is the most successfull IPL team?")
plt.xlabel("Number of  Wins")
plt.ylabel("Teams")
plt.gca().invert_yaxis()
plt.plot()

In [ ]:
deliveries_df

In [ ]:
df_pivot = deliveries_df.pivot_table(index='batter',values='batsman_runs', aggfunc='sum').reset_index()

In [ ]:
top_batsmen = df_pivot.sort_values(by='batsman_runs' , ascending=False).head(20)

In [ ]:
top_batsmen

In [ ]:
plt.barh(top_batsmen['batter'], top_batsmen['batsman_runs'])
plt.xlabel('Total Runs')
plt.ylabel('Batsman')
plt.title("Who is the most performing cricketer?")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
deliveries_df['dismissal_kind'].unique()

In [ ]:
dismissal_not_needed = ['run out', 'retired out', 'obstructing the field', 'retired hurt']

In [ ]:
wickets = deliveries_df[deliveries_df['dismissal_kind'].notna()]

In [ ]:
wickets = wickets[~wickets['dismissal_kind'].isin(dismissal_not_needed)]

In [ ]:
bowler_wickets = wickets.pivot_table(index='bowler', values='is_wicket', aggfunc='sum').reset_index()

In [ ]:
bowler_wickets.rename(columns={'is_wicket':'total_wickets'}, inplace=True)

In [ ]:
top_bowlers = bowler_wickets.sort_values(by='total_wickets', ascending=False).head(20)

In [ ]:
top_bowlers

In [ ]:
plt.barh(top_bowlers['bowler'], top_bowlers['total_wickets'])
plt.xlabel('Total Wickets')
plt.ylabel('Bowler')
plt.title('Who is the most dangerous to bat against?')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
matches_df

In [ ]:
matches_df['toss_win_match'] = matches_df['toss_winner'] == matches_df['winner']

In [ ]:
toss_impact = matches_df['toss_win_match'].value_counts().reset_index()

In [ ]:
toss_impact['toss_win_match'] = toss_impact['toss_win_match'].apply(lambda x: 'Won Match' if x==True else 'Lost Match')

In [ ]:
explodes = [0.01,0]

In [ ]:
plt.pie(toss_impact['count'], labels=toss_impact['toss_win_match'], autopct="%1.2f%%",startangle=90,  colors = ['green', 'red'], explode=explodes)
plt.title("Does winning the toss equals winning the match?", fontsize = 16)
plt.show()

In [ ]:
potm = matches_df['player_of_match'].value_counts().rename_axis('player').reset_index(name='awards').sort_values(by='awards',ascending=False).head(20)

In [ ]:
plt.barh(potm['player'], potm['awards'])
plt.xlabel('Total Awards')
plt.ylabel('Player')
plt.title('Who is the most consistent batsman?')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
potm

In [ ]:
deliveries_df

In [ ]:
batting_stats = deliveries_df.groupby('batter').agg(total_runs=('batsman_runs','sum'),
                                                    total_balls=('batsman_runs','count'),
                                                    innings_played=('match_id','nunique')).reset_index()

In [ ]:
batting_stats['strike_rate'] = (batting_stats['total_runs']/batting_stats['total_balls'])*100

In [ ]:
batting_stats = batting_stats[batting_stats['innings_played']>=20].sort_values(by='innings_played',ascending=False)

In [ ]:
batting_stats['runs_scaled'] = (batting_stats['total_runs'] - batting_stats['total_runs'].min()) / (batting_stats['total_runs'].max() - batting_stats['total_runs'].min())

batting_stats['sr_scaled'] = (batting_stats['strike_rate'] - batting_stats['strike_rate'].min()) / (batting_stats['strike_rate'].max() - batting_stats['strike_rate'].min())

batting_stats['innings_scaled'] = (batting_stats['innings_played'] - batting_stats['innings_played'].min()) / (batting_stats['innings_played'].max() - batting_stats['innings_played'].min())

# Runs = 40%, Strike Rate = 35%, Consistency = 25%
batting_stats['performance_score'] = (
    batting_stats['runs_scaled'] * 0.40 +
    batting_stats['sr_scaled'] * 0.35 +
    batting_stats['innings_scaled'] * 0.25
)


top_performers = batting_stats.sort_values(by='performance_score', ascending=False).head(15)

top_performers[['batter', 'total_runs','total_balls','innings_played', 'strike_rate', 'performance_score']]

In [ ]:
plt.barh(top_performers['batter'], top_performers['performance_score'])
plt.gca().invert_yaxis()
plt.title('Top 15 IPL Batsmen - Overall Performance Score', fontsize=16)
plt.xlabel('Performance Score (0-1)')
plt.ylabel('Batsman')
plt.axvline(x=0.7, color='red', linestyle='--', label='Elite Threshold')
plt.legend()
plt.show()

In [ ]:
matches_df.to_csv('matches_cleaned.csv', index=False)
deliveries_df.to_csv('deliveries_cleaned.csv', index=False)
batting_stats.to_csv('batting_stats.csv', index=False)


In [ ]:
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

db_password = getpass("Enter PostgreSQL password: ")

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=db_password,
    host="localhost",
    port=5432,
    database="ipl_project"
)

engine = create_engine(connection_url)

matches_df.to_sql("matches", engine, if_exists="replace", index=False)
deliveries_df.to_sql("deliveries", engine, if_exists="replace", index=False)
batting_stats.to_sql("batting_stats", engine, if_exists="replace", index=False)

print("Tables created successfully!")

In [ ]:
matches_main = pd.DataFrame(matches)

In [ ]:
deliveries_df_with_seasons = deliveries_df.merge(matches_main[['id', 'season']],
                                                 left_on='match_id', right_on='id')


In [ ]:
batting_stats_altered = deliveries_df_with_seasons.groupby(['batter','season']).agg(Total_runs = ('batsman_runs','sum'),
                                                                                    Total_balls= ('batsman_runs','count'),
                                                                                    Innings_played=('match_id','nunique')).reset_index()

In [ ]:
batting_stats_altered['strike_rate']=(batting_stats_altered['Total_runs']/batting_stats_altered['Total_balls'])*100

In [ ]:
temp_stats = batting_stats_altered.groupby('batter').agg(
    innings_played=('Innings_played', 'sum')
).reset_index()

In [ ]:
temp_stats = temp_stats.sort_values('innings_played',ascending=False)

In [ ]:
temp_stats=temp_stats[temp_stats['innings_played']>=20]

In [ ]:
batting_stats_altered = batting_stats_altered[batting_stats_altered['batter'].isin(temp_stats['batter'])]

In [ ]:
batting_stats_altered['runs_scaled'] = (batting_stats_altered['Total_runs'] - batting_stats_altered['Total_runs'].min()) / (batting_stats_altered['Total_runs'].max() - batting_stats_altered['Total_runs'].min())

batting_stats_altered['sr_scaled'] = (batting_stats_altered['strike_rate'] - batting_stats_altered['strike_rate'].min()) / (batting_stats_altered['strike_rate'].max() - batting_stats_altered['strike_rate'].min())

batting_stats_altered['innings_scaled'] = (batting_stats_altered['Innings_played'] - batting_stats_altered['Innings_played'].min()) / (batting_stats_altered['Innings_played'].max() - batting_stats_altered['Innings_played'].min())

# Runs = 40%, Strike Rate = 35%, Consistency = 25%
batting_stats_altered['performance_score'] = (
    batting_stats_altered['runs_scaled'] * 0.40 +
    batting_stats_altered['sr_scaled'] * 0.35 +
    batting_stats_altered['innings_scaled'] * 0.25
)

In [ ]:
deliveries_df_with_seasons

In [ ]:
filtered_wickets = deliveries_df[~deliveries_df['dismissal_kind'].isin(dismissal_not_needed)]

In [ ]:
bowling_stats = filtered_wickets.groupby('bowler').agg(total_wickets = ('is_wicket','sum'),
                                                       total_balls = ('ball','count'),
                                                       runs_conceded = ('total_runs','sum'),
                                                       matches_played = ('match_id','nunique')
                                                       )

In [ ]:
bowling_stats['economy_rate'] = (bowling_stats['runs_conceded'] / bowling_stats['total_balls']) * 6 

In [ ]:
bowling_stats['bowling_average'] = bowling_stats['runs_conceded'] / bowling_stats['total_wickets'].replace(0,1)

In [ ]:
bowling_stats = bowling_stats.reset_index()

In [ ]:
bowling_stats.sort_values(by='total_wickets', ascending=False) 

In [ ]:
filtered_wickets_with_season = filtered_wickets.merge(
    matches_main[['id', 'season']], 
    left_on='match_id', 
    right_on='id',
    how='left'
)


In [ ]:
bowling_stats_improved = filtered_wickets_with_season.groupby(['bowler', 'season']).agg(
    total_wickets=('is_wicket', 'sum'),
    total_balls=('ball', 'count'),
    runs_conceded=('total_runs', 'sum'),
    matches_played=('match_id', 'nunique')
).reset_index()

In [ ]:
bowling_stats_improved['economy_rate'] = (bowling_stats_improved['runs_conceded'] / bowling_stats_improved['total_balls']) * 6
bowling_stats_improved['bowling_average'] = bowling_stats_improved['runs_conceded'] / bowling_stats_improved['total_wickets'].replace(0, 1)

In [ ]:
bowling_stats_improved

In [ ]:
dismissal_stats = filtered_wickets.groupby(['bowler','dismissal_kind']).agg(wickets = ('is_wicket','count')).reset_index()

In [ ]:
dismissal_stats_improved =filtered_wickets_with_season.groupby(
    ['bowler', 'dismissal_kind', 'season']).agg(wickets=('is_wicket', 'count')).reset_index()

dismissal_stats_improved.to_csv('dismissal_stats_improved.csv', index=False)

In [ ]:
bowling_stats.to_csv('bowling_stats.csv',index=False)
dismissal_stats.to_csv('dismissal_stats.csv',index=False)
bowling_stats_improved.to_csv('bowling_stats_improved.csv',index=False)